# 2022 韩国关税署原稿的互联网档案索引

只读官网页与HWP确切URL在2022–2023年的CDX捕获索引；先存响应和NAS，不把索引条目当作原件内容。

In [1]:
import hashlib,json,subprocess,tempfile
from datetime import datetime,timezone,date
from pathlib import Path
from src.data.nas_market_data import publish_nas_snapshot,materialize_nas_snapshot
root=Path.cwd();card=root/'research/kcs_early20_2022_wayback_vintage_probe_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists()
control=root/plan['source_control']['card'];assert sha(control/'summary.json')==plan['source_control']['summary_sha256'] and json.loads((control/'source_review.json').read_text())['HWP_sha256']==plan['source_control']['HWP_current_sha256']
raw=root/'data/snapshots/kcs_early20_2022_wayback_vintage_probe_20261023';raw.mkdir(parents=True,exist_ok=True);assert not any(raw.iterdir())
state={'status':'started','plan_sha256':sha(card/'plan.json'),'GET_attempts':0,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];results=[];t=plan['transport']
for spec in plan['exact_requests']:
    state['GET_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
    body=raw/(spec['name']+'.body');headers=raw/(spec['name']+'.headers');started=datetime.now(timezone.utc).isoformat()
    cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(t['connect_timeout_seconds']),'--max-time',str(t['total_timeout_seconds_each']),'--max-filesize',str(t['max_response_bytes_each']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',t['user_agent'],'-w','%{http_code} %{url_effective}',spec['url']]
    try:
        run=subprocess.run(cmd,capture_output=True,text=True,timeout=35);parts=run.stdout.split(' ',1);rec={'name':spec['name'],'target_url':spec['target_url'],'request_url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':int(parts[0]) if parts[0].isdigit() else None,'final_url':parts[1] if len(parts)>1 else None,'transport_error':run.stderr[:250]}
    except Exception as exc:rec={'name':spec['name'],'target_url':spec['target_url'],'request_url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':type(exc).__name__+':'+str(exc)[:180]}
    rec.update(body_sha256=sha(body) if body.exists() else None,body_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0)
    receipts.append(rec);(raw/'capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
    item={'name':spec['name'],'target_url':spec['target_url'],'status':'transport_failure','captures':[]}
    if rec.get('curl_exit_code')==0 and rec.get('http_status')==200 and body.exists() and body.stat().st_size<=t['max_response_bytes_each']:
        try:
            data=json.loads(body.read_text());assert isinstance(data,list) and data and isinstance(data[0],list)
            header=data[0];required={'timestamp','original','statuscode','digest'};assert required.issubset(header)
            rows=[dict(zip(header,row)) for row in data[1:]];assert all(len(row)==len(header) for row in data[1:])
            valid=[x for x in rows if x['statuscode']=='200' and x['timestamp'][:4] in ('2022','2023')]
            item.update(status='CDX_rows_received' if valid else 'CDX_no_matching_rows',captures=valid,CDX_row_count=len(rows),CDX_fields=header)
        except Exception as exc:item.update(status='CDX_schema_failure',reason=type(exc).__name__+':'+str(exc)[:160])
    results.append(item)
assert state['GET_attempts']==2
manifest={'schema_version':1,'experiment':plan['experiment'],'plan_sha256':state['plan_sha256'],'capture_utc':datetime.now(timezone.utc).isoformat(),'receipts':receipts,'results':results,'historical_first_public_bytes_qualified':False}
(raw/'source_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2)+'\n');(card/'source_receipt.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2)+'\n')
files={str(p.relative_to(root)):'raw' for p in raw.iterdir() if p.is_file()}
try:
    binding=publish_nas_snapshot(workspace_root=root,source_manifest=raw/'source_manifest.json',files=files,as_of=date.today().isoformat(),source_metadata={'experiment':plan['experiment'],'source':'Internet Archive CDX KCS 2022 URL vintage index'})
    (card/'nas_binding.json').write_text(json.dumps(binding,indent=2)+'\n')
    with tempfile.TemporaryDirectory(prefix='kcs-2022-cdx-cold-') as temp:
        readback=materialize_nas_snapshot(binding,workspace_root=Path(temp))
        for obj in binding['objects']:assert sha(Path(temp)/obj['local_path'])==obj['sha256']
    assert binding['verified'] and readback['restored_objects']==len(files)
    nas_ok=True
except Exception as exc:nas_ok=False;binding=None;manifest['NAS_failure']=type(exc).__name__+':'+str(exc)[:180]
state.update(status='completed' if nas_ok else 'stopped',NAS_readback_verified=nas_ok);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
result={'decision':'Internet_Archive_CDX_2022_KCS_exact_URL_vintage_probe_source_only' if nas_ok else 'needs_CDX_NAS_backup','GET_attempts':2,'results':results,'NAS_source_id':binding['source_id'] if binding else None,'NAS_readback_verified':nas_ok,'historical_first_public_bytes_qualified':False,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n');print(json.dumps({'decision':result['decision'],'results':results,'NAS_readback_verified':nas_ok},ensure_ascii=False))


{"decision": "Internet_Archive_CDX_2022_KCS_exact_URL_vintage_probe_source_only", "results": [{"name": "official_page", "target_url": "https://www.korea.kr/briefing/pressReleaseView.do?newsId=156531875", "status": "CDX_schema_failure", "captures": [], "reason": "AssertionError:"}, {"name": "official_HWP", "target_url": "https://www.korea.kr/common/download.do?fileId=197006618&tblKey=GMN", "status": "CDX_schema_failure", "captures": [], "reason": "AssertionError:"}], "NAS_readback_verified": true}


# 2022 年韩国关税署原稿的独立历史快照探测

按冻结的两个**确切 URL**，分别查询互联网档案馆 CDX 对韩国政府 2022-10-21 原稿页及其 HWP 附件在 **2022–2023 年**的 HTTP 200 捕获。两次请求均得到 HTTP 200、`application/json`、原始内容 `[]\n`（各 3 字节）；原始响应和 UTC 收据已进入 Yanxuan NAS 并远端核对、定向恢复。首次解析把无表头的空数组误记作 schema 失败，保存原结果后，仅用相同原件作一次解释修正；详情见 `empty_array_interpretation.json`。

结论严格限于：**这两个 URL 在此次固定 CDX 条件下没有返回捕获记录**，因而没有可比对的当时 HWP 字节。它不证明其他规范化 URL、年份或其他档案馆均无记录，也不否定政府页面和 HWP 内印的 2022-10-21 09:00 KST 分发时间；后者仍不是独立首次在线收据。当前 2022 附件的半导体 **5,573 百万美元、同比 −12.8%** 数值来源已核对，但不能因此赋予完整历史 PIT 资格。

本卡共 2 次 CDX GET、0 SOXX 结果读取、0 拟合、0 账户。韩国前 20 日历史源仍缺逐月首次公开版本、修订与美股决策前可用时点，不能推动 Stage1 0bp 策略晋升。
